# Model Evaluation — Real vs. Physics-Informed vs. GAN Synthetic Data

This notebook puts every regression model currently in this project
(`model_registry.MODEL_FACTORIES`) through the same test: **given a
voltammogram signal, how well does the model recover the concentration?**

## Models under test

| Key | Algorithm | Input | Hyperparameter source |
|---|---|---|---|
| `ridge` | Ridge regression | engineered features | `full_range_data_augmentation.ipynb` ablation study |
| `random_forest` | Random Forest | engineered features | ablation study |
| `xgboost` | XGBoost | engineered features | ablation study |
| `svr` | SVR (RBF) | engineered features | `training/tune_ml_hparams.py` (nested LOOCV + `skopt.BayesSearchCV`) |
| `xgboost_tuned` | XGBoost | engineered features | `training/tune_ml_hparams.py` |
| `mlp` | MLP | engineered features | `training/tune_dl_hparams.py` (Optuna) |
| `cnn` | 1D-CNN | **raw 229-point signal** | `training/tune_dl_hparams.py` (Optuna) |

`svr` / `xgboost_tuned` / `mlp`'s hyperparameters were searched specifically
against the `experimental` (23-feature) suite. Reusing those same
hyperparameter *values* on `core`/`extended` below is a same-hyperparameters,
different-inputs comparison, not a suite-specific re-tuning - a fully
rigorous per-suite comparison would mean re-running
`tune_ml_hparams.py`/`tune_dl_hparams.py` once per suite, which is out of
scope here. `cnn` has no suite dimension at all - it always trains on the
raw signal.

## Data sources under test

1. **Physics-informed augmented data** (`raw/raw_signals_augmented.csv`,
   generated by `augmentation_pipeline.py`'s Long & Winefordner noise model +
   PCHIP Ip-concentration spline).
2. **GAN-synthetic data**, both architectures, both training regimes:
   `wgangp` / `timegan` x trained-on-real / trained-on-combined
   (`gan_output/samples_trained_on_*`).
3. **Real data** (`raw/raw_signals_real.csv`) - see the note on leakage below.

## Handling data leakage

Every model here is first fit **once, on the real 40 signals only**
(mirrors `export_models.py`'s own real-data-only baseline philosophy) and
then tested against (1) and (2) above. Since neither the physics-informed
nor the GAN data was seen during that fit, those two tests are leakage-free
by construction - no special handling needed.

Testing on the real data itself is the one case that needs care: those same
40 signals were used to fit the very models being tested, so a naive test
would just measure memorization.
`full_range_data_augmentation.ipynb` (Section 10, Ablation Study) already
solves this with **Protocol C**: hold out one real signal, train on the
*rest* of the real signals plus *all* the physics-augmented ones, test on
the held-out real signal, repeat LOOCV-style over all 40. That's exactly
what Section 6 below does, for every model and suite - it's what makes a
real-data comparison possible at all without leaking.

(`model_training.ipynb`'s nested LOOCV, and `tune_ml_hparams.py`'s reuse of
that same methodology, is a related but different tool: it's a
hyperparameter-search generalisation estimate on real data *alone*, not a
"does synthetic training data help" comparison. Protocol C's held-out real
signal is also *not* the "reference" reading elsewhere in this project - it
never trains on synthetic data alone.)


## 1 · Imports

In [ ]:

import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut

import data_registry as dr
from augmentation_pipeline import FEATURE_COLUMNS_BY_SUITE
from model_registry import MODEL_FACTORIES, MODEL_LABELS, RAW_SIGNAL_MODELS, predict, train_models
from plot_style import apply_default_plotly_layout

SUITES = ['core', 'extended', 'experimental']  # tabular suites; 'cnn' always uses 'raw_signal'
TABULAR_MODEL_KEYS = [k for k in MODEL_FACTORIES if k not in RAW_SIGNAL_MODELS]

# Fixed per-model color, reused across every chart in this notebook (validated
# for categorical CVD-safety with dataviz's palette validator).
MODEL_COLORS = {
    'ridge': '#17becf', 'random_forest': '#e34a1a', 'xgboost': '#5a23c4',
    'svr': '#2ca02c', 'xgboost_tuned': '#c9a227', 'mlp': '#0b5fa5', 'cnn': '#a83279',
}


## 2 · Load every data source

All sources share the same 229-point potential grid and on-disk row format
(`concentration`, `I_0` .. `I_228`), so one loader works for all of them.


In [ ]:

def load_signal_csv(path: str) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    y = df['concentration'].to_numpy(dtype=float)
    X = df.drop(columns=['concentration']).to_numpy(dtype=float)
    return X, y


E = pd.read_csv('raw/raw_potential_grid.csv')['potential_V'].to_numpy(dtype=float)

X_real, y_real = load_signal_csv('raw/raw_signals_real.csv')
X_aug, y_aug = load_signal_csv('raw/raw_signals_augmented.csv')

GAN_SOURCE_PATHS = {
    'wgangp_real':      'gan_output/samples_trained_on_real/wgangp_signals.csv',
    'wgangp_combined':  'gan_output/samples_trained_on_combined/wgangp_signals.csv',
    'timegan_real':     'gan_output/samples_trained_on_real/timegan_signals.csv',
    'timegan_combined': 'gan_output/samples_trained_on_combined/timegan_signals.csv',
}
gan_data = {name: load_signal_csv(path) for name, path in GAN_SOURCE_PATHS.items()}

print(f'real signals:              {len(y_real)}')
print(f'physics-augmented signals: {len(y_aug)}')
for name, (_, y) in gan_data.items():
    print(f'{name:<18} signals: {len(y)}')


## 3 · Featurization and metrics helpers

In [ ]:

def featurize(X: np.ndarray, y: np.ndarray, suite: str) -> pd.DataFrame:
    return dr.featurize(E, X, y, suite=suite)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return dict(
        MAE=float(mean_absolute_error(y_true, y_pred)),
        RMSE=float(np.sqrt(mean_squared_error(y_true, y_pred))),
        R2=float(r2_score(y_true, y_pred)),
        n=int(len(y_true)),
    )


## 4 · Fit every model, once, on real data only

Mirrors `export_models.py`: one fit per (model, suite) on all 40 real
signals - not a search, not a nested CV - "the model as it would actually
ship." These are the models Section 5 tests against physics-augmented and
GAN data below.


In [ ]:

def fit_all_models(X: np.ndarray, y: np.ndarray) -> dict:
    # Returns {(model_name, suite): (fitted_model, imputer, feature_columns)}.
    fitted = {}
    for suite in SUITES:
        feat_df = featurize(X, y, suite)
        cols = FEATURE_COLUMNS_BY_SUITE[suite]
        X_feat, y_feat = feat_df[cols], feat_df['concentration']
        models, imputer, _ = train_models(X_feat, y_feat, TABULAR_MODEL_KEYS)
        for name, model in models.items():
            fitted[(name, suite)] = (model, imputer, cols)

    raw_cols = FEATURE_COLUMNS_BY_SUITE['raw_signal']
    raw_feat_df = featurize(X, y, 'raw_signal')
    X_raw, y_raw = raw_feat_df[raw_cols], raw_feat_df['concentration']
    cnn_models, _, raw_imputer = train_models(X_raw, y_raw, ['cnn'], X_raw_signal=X_raw)
    fitted[('cnn', 'raw_signal')] = (cnn_models['cnn'], raw_imputer, raw_cols)
    return fitted


t0 = time.time()
real_fitted = fit_all_models(X_real, y_real)
print(f'Fitted {len(real_fitted)} (model, suite) combinations on {len(y_real)} real signals '
      f'in {time.time() - t0:.1f}s')


## 5 · Test against physics-augmented and GAN data (no leakage)

Every model above was fit on real data alone, so testing it against
synthetic data it never saw is leakage-free by construction.


In [ ]:

def evaluate_against(fitted: dict, X_test: np.ndarray, y_test: np.ndarray, test_source: str) -> pd.DataFrame:
    rows = []
    for (model_name, suite), (model, imputer, cols) in fitted.items():
        feat_df = featurize(X_test, y_test, suite)
        X_feat = feat_df[cols]
        y_true = feat_df['concentration'].to_numpy(dtype=float)
        y_pred = predict(model, imputer, X_feat)
        rows.append(dict(model=model_name, suite=suite, test_source=test_source,
                          **regression_metrics(y_true, y_pred)))
    return pd.DataFrame(rows)


no_leak_frames = [evaluate_against(real_fitted, X_aug, y_aug, 'physics_augmented')]
for name, (X, y) in gan_data.items():
    no_leak_frames.append(evaluate_against(real_fitted, X, y, name))

no_leak_results = pd.concat(no_leak_frames, ignore_index=True)
no_leak_results.sort_values(['test_source', 'suite', 'MAE']).reset_index(drop=True)


## 6 · Test against real data — without leakage (Protocol C)

For each of the 40 real signals: fit on the *other* 39 real signals plus
*all* physics-augmented ones, predict the held-out signal, repeat LOOCV-style.
The held-out signal is never part of its own model's training data, so this
is a genuine (if small-sample) generalisation estimate on real data - unlike
naively testing the Section 4 models (which already saw every real signal).

This runs `n_folds x n_models x n_suites` fits, so it's the slow cell in
this notebook - the sklearn/XGBoost models are fast, but `mlp` and `cnn`
each retrain a small network per fold. Expect this to take a while
(single-digit minutes) for the 40 folds x 6 tabular models x 3 suites + 40
CNN folds).


In [ ]:

def protocol_c_real(model_name: str, suite: str) -> dict:
    loo = LeaveOneOut()
    is_raw = model_name in RAW_SIGNAL_MODELS
    cols = FEATURE_COLUMNS_BY_SUITE[suite]
    y_true_all, y_pred_all = [], []

    for train_idx, test_idx in loo.split(X_real):
        X_train = np.concatenate([X_real[train_idx], X_aug], axis=0)
        y_train = np.concatenate([y_real[train_idx], y_aug], axis=0)

        train_feat_df = featurize(X_train, y_train, suite)
        X_feat, y_feat = train_feat_df[cols], train_feat_df['concentration']
        if is_raw:
            models, _, imputer = train_models(X_feat, y_feat, [model_name], X_raw_signal=X_feat)
        else:
            models, imputer, _ = train_models(X_feat, y_feat, [model_name])

        test_feat_df = featurize(X_real[test_idx], y_real[test_idx], suite)
        pred = predict(models[model_name], imputer, test_feat_df[cols])

        y_true_all.append(y_real[test_idx][0])
        y_pred_all.append(pred[0])

    return regression_metrics(np.array(y_true_all), np.array(y_pred_all))


t0 = time.time()
protocol_c_rows = []
for model_name in TABULAR_MODEL_KEYS:
    for suite in SUITES:
        m = protocol_c_real(model_name, suite)
        protocol_c_rows.append(dict(model=model_name, suite=suite, test_source='real_protocol_c', **m))
        print(f'  {model_name:<14} {suite:<12} MAE={m["MAE"]:.3f}  ({time.time() - t0:.0f}s elapsed)')

m = protocol_c_real('cnn', 'raw_signal')
protocol_c_rows.append(dict(model='cnn', suite='raw_signal', test_source='real_protocol_c', **m))
print(f'  {"cnn":<14} {"raw_signal":<12} MAE={m["MAE"]:.3f}  ({time.time() - t0:.0f}s elapsed)')

protocol_c_results = pd.DataFrame(protocol_c_rows)
print(f'\nProtocol C done in {time.time() - t0:.0f}s total')
protocol_c_results.sort_values(['suite', 'MAE']).reset_index(drop=True)


## 7 · Combined results table

In [ ]:

all_results = pd.concat([no_leak_results, protocol_c_results], ignore_index=True)
all_results['model_label'] = all_results['model'].map(MODEL_LABELS)
all_results.to_csv('results/model_evaluation_results.csv', index=False)
all_results.sort_values(['test_source', 'suite', 'MAE']).reset_index(drop=True)


## 8 · Visualisation

Two comparisons, both on the `experimental` suite (the one every tuned
model's hyperparameters actually target):

1. **MAE by test source** - how does each model's error change across
   physics-augmented data, each GAN variant, and the leakage-free real-data
   read (Protocol C)?
2. **MAE by feature suite** - does the extra feature complexity in
   `extended`/`experimental` actually help, for a fixed test source?


In [ ]:

def plot_mae_by_test_source(df: pd.DataFrame, suite: str = 'experimental') -> go.Figure:
    sub = df[(df['suite'] == suite) | (df['model'] == 'cnn')]
    test_sources = list(dict.fromkeys(sub['test_source']))  # stable order, de-duped

    fig = go.Figure()
    for model_name, label in MODEL_LABELS.items():
        row = sub[sub['model'] == model_name].set_index('test_source').reindex(test_sources)
        fig.add_trace(go.Bar(x=test_sources, y=row['MAE'], name=label,
                              marker_color=MODEL_COLORS[model_name]))
    fig.update_layout(barmode='group')
    return apply_default_plotly_layout(
        fig, title_text=f'MAE by test source ({suite} suite, cnn always raw signal)',
        xaxis_title='Test source', yaxis_title='MAE (uM)')


def plot_mae_by_suite(df: pd.DataFrame, test_source: str = 'physics_augmented') -> go.Figure:
    sub = df[(df['test_source'] == test_source) & (df['model'] != 'cnn')]

    fig = go.Figure()
    for model_name in TABULAR_MODEL_KEYS:
        row = sub[sub['model'] == model_name].set_index('suite').reindex(SUITES)
        fig.add_trace(go.Bar(x=SUITES, y=row['MAE'], name=MODEL_LABELS[model_name],
                              marker_color=MODEL_COLORS[model_name]))
    fig.update_layout(barmode='group')
    return apply_default_plotly_layout(
        fig, title_text=f'MAE by feature suite - {test_source}',
        xaxis_title='Feature suite', yaxis_title='MAE (uM)')


plot_mae_by_test_source(all_results, suite='experimental').show()


In [ ]:

plot_mae_by_suite(all_results, test_source='physics_augmented').show()


## 9 · Discussion

Fill in after running the full notebook (Section 6 in particular takes a
while) - a few questions the results above are set up to answer:

- Which model generalises best to each synthetic source - is it the same
  model across physics-augmented and both GAN architectures, or does the
  ranking flip?
- Does `xgboost_tuned` actually beat the ablation-study `xgboost` here, on
  data neither was tuned against - or did the LOOCV+BayesSearchCV tuning
  overfit to the real-only, `experimental`-suite setting it was found in?
- Does the 1D-CNN's raw-signal approach hold up against the engineered-
  feature models on GAN-generated data, where the "signal" is itself a
  model's output rather than a physical measurement?
- Does the `core` -> `extended` -> `experimental` suite progression help,
  hurt, or plateau, per model and per test source?
- How does the Protocol C real-data MAE compare to the physics-augmented
  and GAN MAEs for the same model - is real data still the hardest test, or
  has synthetic data caught up?
